Assignment 2 Stage 2

This notebook loads the exact model checkpoint submitted in Stage 1 and evaluates it on the hidden test dataset.


In [1]:
import pandas as pd
import numpy as np
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
hidden_df = pd.read_csv("hidden_test_with_labels.csv")

print("Hidden test shape:", hidden_df.shape)
hidden_df.head()

Hidden test shape: (600, 5)


,id,text,label,label_name,source_file
0,neg_cv795_10291,"mr . bean , a bumbling security guard from eng...",0,negative,neg/cv795_10291.txt
1,neg_cv174_9735,"starship troopers is a bad movie . \ni mean , ...",0,negative,neg/cv174_9735.txt
2,pos_cv065_15248,"what a great film . \nwhat a stunning , touchi...",1,positive,pos/cv065_15248.txt
3,neg_cv076_26009,"susan granger's review of "" the watcher "" ( un...",0,negative,neg/cv076_26009.txt
4,neg_cv417_14653,the marvelous british actor derek jacobi stars...,0,negative,neg/cv417_14653.txt


In [3]:
checkpoint = torch.load(
    "model_checkpoint/model_checkpoint.pt",
    map_location="cpu"
)

word_to_id = checkpoint["word_to_id"]
MAX_LENGTH = checkpoint["max_length"]

print("Vocabulary size:", len(word_to_id))
print("Max length:", MAX_LENGTH)

Vocabulary size: 7305
Max length: 300


In [4]:
import re

def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-Z']+", text)

PAD_ID = word_to_id["<PAD>"]
UNK_ID = word_to_id["<UNK>"]

def encode_review(text):
    tokens = tokenize(text)
    ids = [word_to_id.get(token, UNK_ID) for token in tokens]

    ids = ids[:MAX_LENGTH]

    if len(ids) < MAX_LENGTH:
        ids += [PAD_ID] * (MAX_LENGTH - len(ids))

    return ids

In [5]:
class HiddenReviewDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = encode_review(self.texts[idx])

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )


hidden_dataset = HiddenReviewDataset(hidden_df)

hidden_loader = DataLoader(
    hidden_dataset,
    batch_size=32,
    shuffle=False
)

print("Hidden test samples:", len(hidden_dataset))

Hidden test samples: 600


In [6]:
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)
        _, hidden = self.gru(embeddings)

        final_hidden = hidden[-1]
        final_hidden = self.dropout(final_hidden)

        return self.output_layer(final_hidden)

In [7]:
model = SentimentGRU(
    vocab_size=len(word_to_id),
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"]
)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Stage 1 model loaded successfully.")

Stage 1 model loaded successfully.


In [8]:
all_predictions = []
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in hidden_loader:
        logits = model(x_batch)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(predictions.numpy())
        all_labels.extend(y_batch.numpy())

hidden_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

hidden_confusion_matrix = confusion_matrix(
    all_labels,
    all_predictions
)

print("Hidden test accuracy:", hidden_accuracy)
print("\nHidden test confusion matrix:")
print(hidden_confusion_matrix)

Hidden test accuracy: 0.49166666666666664

Hidden test confusion matrix:
[[ 55 245]
 [ 60 240]]


In [9]:
predictions_df = pd.DataFrame({
    "id": hidden_df["id"],
    "predicted_label": all_predictions
})

predictions_df.to_csv(
    "hidden_test_predictions.csv",
    index=False
)

print(predictions_df.head())
print("\nNumber of predictions:", len(predictions_df))

                id  predicted_label
0  neg_cv795_10291                1
1   neg_cv174_9735                1
2  pos_cv065_15248                1
3  neg_cv076_26009                1
4  neg_cv417_14653                1

Number of predictions: 600


Stage 2 Results


The same model checkpoint that was submitted in stage one was revaulated on the hidden test dataset without changing the model and the results are as shown below:

- Hidden test accuracy: 49.17%
- Hidden test samples: 600
- Confusion matrix:

[[55, 245],
 [60, 240]]

The model was shows to perform worse on the hidden test set than on the public provided one and the confusion matrix also shows that the model was able to predict positive reviews more often than the negative ones.

If I had the chance or more time i would play around with different wordings or work on the model understanding words and connotations better or even using different models.

Use of AI

I used generative AI as a learning and programming assistant during Stage 2. It helped me understand how to reload the Stage 1 checkpoint and perform any of the inference without retraining or altering the models. I used propmts such as “How do I use my saved Stage 1 vocabulary to process the hidden test reviews?” and “What does my hidden test confusion matrix mean?” to evaluate and understand my results. All reported results were produced by running the code in this notebook.